![test](https://external-content.duckduckgo.com/iu/?u=http%3A%2F%2Fdrive.google.com/uc?id=1uKZyIFt0sOAFgUgMNf7BS1GVrtaB-s_p)

## **Created for**

**Environment:** `nvcr.io/nvidia/pytorch:24.10-py3`  
**GPU:** 18 GB MIG-Partitioned GPU

# **Vector Database Benchmarking: FAISS vs ChromaDB vs Qdrant**

This notebook benchmarks three widely-used options - FAISS, ChromaDB, and Qdrant - across multiple embedding models on the SciFact dataset, measuring each system on build time, query latency, and Recall@10.


## **What Are Vector Databases?**

A vector database stores high-dimensional embeddings and enables fast approximate or exact nearest-neighbor search over them. Unlike traditional databases that match on exact values, vector databases retrieve results based on geometric similarity - making them the core retrieval layer in semantic search, RAG pipelines, and recommendation systems. Each database trades off differently between search speed, memory footprint, persistence, and metadata support.


## **What This Notebook Covers**

- Load precomputed embeddings for three models - `MiniLM`, `BGE`, and `E5` - generated in Notebook 1
- Build indexes and collections in FAISS, ChromaDB, and Qdrant and record construction time
- Benchmark per-query latency across all three databases
- Evaluate retrieval quality using Recall@10 against SciFact ground truth relevance labels
- Compare results across databases and embedding models to identify the best-performing combinations

## **Pinning Numpy to 1.26.4**
Uninstalling numpy if present and installing the `1.26.4` version

In [ ]:
!pip uninstall numpy -y

In [ ]:
pip install --upgrade --force-reinstall numpy==1.26.4

# **Kernel Restart Required**

The code uses IPython's application instance to programmatically trigger a kernel shutdown with restart enabled — equivalent to clicking **Kernel > Restart** manually, but automated inline.

- `IPython.Application.instance()` — fetches the currently running IPython kernel process
- `app.kernel.do_shutdown(True)` — sends a shutdown signal; `True` tells it to restart instead of terminate

No reinstallation is needed. The library is already present on the system and will be available immediately after the kernel restarts. No need to re-run above cells, you may continue from the cell below.

In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

# **Import Required Libraries**

All the required libraries are imported here

In [ ]:
import os
import time
import pickle
import numpy as np
import pandas as pd

from tqdm import tqdm

import faiss
import chromadb

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)
from beir import util
from beir.datasets.data_loader import GenericDataLoader

## **Load Embeddings Generated in Notebook 1**

Loads precomputed document embeddings from disk.

- `embedding_store` - central dict keyed by model name, allowing uniform access to embeddings throughout the notebook
- Each `.npy` file contains the full corpus embeddings generated in Notebook 1 for `MiniLM`, `BGE`, and `E5`
- `np.load()` deserializes the arrays directly into memory with their original shape and dtype preserved


In [ ]:
embedding_store = {}

embedding_store["MiniLM"] = np.load(
    "saved_embeddings/MiniLM.npy"
)

embedding_store["BGE"] = np.load(
    "saved_embeddings/BGE.npy"
)

embedding_store["E5"] = np.load(
    "saved_embeddings/E5.npy"
)

In [ ]:
with open(
    "saved_embeddings/documents.pkl",
    "rb"
) as f:

    documents = pickle.load(f)

In [ ]:
with open(
    "saved_embeddings/doc_ids.pkl",
    "rb"
) as f:

    doc_ids = pickle.load(f)

## **Load Ground Truth Labels**

Loads SciFact datase and its three core components for retrieval evaluation.

- `dataset` - set to `"scifact"`, a scientific fact-checking corpus commonly used as a retrieval benchmark
- `util.download_and_unzip()` - fetches and extracts the dataset archive into `./datasets` if not already present
- `GenericDataLoader` - BEIR's unified loader; `split="test"` returns only the evaluation partition
- Returns three objects:
  - `corpus` - dict of document IDs to passage text
  - `queries` - dict of query IDs to query strings
  - `qrels` - ground truth relevance judgments mapping query IDs to relevant document IDs

In [ ]:
dataset = "scifact"

url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

data_path = util.download_and_unzip(
    url,
    "./datasets"
)

corpus, queries, qrels = GenericDataLoader(
    data_folder=data_path
).load(split="test")

## **Create FAISS Indexes**

FAISS (Facebook AI Similarity Search) is a library optimized for dense vector search over large corpora. It stores embeddings in an index structure and retrieves the nearest neighbors using distance metrics like inner product or L2. `IndexFlatIP` performs exact search using inner product, which is equivalent to cosine similarity when vectors are L2-normalized. FAISS operates entirely in-memory and benefits significantly from GPU acceleration, making it well-suited for low-latency retrieval benchmarks.

Builds a separate `IndexFlatIP` index for each embedding model and records the time taken:

- `faiss.IndexFlatIP` - exact inner product search index; dimensionality is inferred from `embeddings.shape[1]`
- `faiss.normalize_L2()` - normalizes vectors in-place so inner product becomes equivalent to cosine similarity
- `index.add()` - loads all corpus embeddings into the index; requires `float32` dtype
- `build_time` - wall-clock duration of the index construction, stored for later comparison
- Results are accumulated in `faiss_metrics` as a list of dicts for easy conversion to a DataFrame


In [ ]:
faiss_indexes = {}

faiss_metrics = []

In [ ]:
for model_name, embeddings in embedding_store.items():

    print(f"\nBuilding FAISS for {model_name}")

    start = time.time()

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(
        dimension
    )

    faiss.normalize_L2(
        embeddings
    )

    index.add(
        embeddings.astype("float32")
    )

    build_time = time.time() - start

    faiss_indexes[model_name] = index

    faiss_metrics.append({
        "Model": model_name,
        "DB": "FAISS",
        "Build Time": build_time
    })

## **Create Chroma Collections**


ChromaDB is an embedded vector database designed for simplicity and local experimentation. Unlike FAISS, it persists data to disk and stores both embeddings and raw documents together in named collections, making it easy to retrieve context alongside search results without managing separate data structures.

Initializes a persistent Chroma client and builds one collection per embedding model:

- `PersistentClient` - writes the database to `./chroma_db`, so collections survive kernel restarts
- `delete_collection()` wrapped in `try/except` - ensures a clean slate by dropping any existing collection with the same name without raising an error if it doesn't exist
- `create_collection()` - creates a fresh collection named `{model_name}_collection`
- `collection.add()` - ingests embeddings alongside raw document text and integer string IDs; Chroma requires embeddings as a Python list rather than a NumPy array, hence `.tolist()`
- `build_time` is appended to the shared `faiss_metrics` list under `"DB": "Chroma"` for unified comparison later

In [ ]:
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

In [ ]:
chroma_collections = {}

In [ ]:
for model_name, embeddings in embedding_store.items():

    collection_name = (
        f"{model_name.lower()}_collection"
    )

    try:
        chroma_client.delete_collection(
            collection_name
        )
    except:
        pass

    collection = chroma_client.create_collection(
        collection_name
    )

    start = time.time()

    collection.add(
        ids=[str(i) for i in range(len(documents))],
        embeddings=embeddings.tolist(),
        documents=documents
    )

    build_time = time.time() - start

    chroma_collections[
        model_name
    ] = collection

    faiss_metrics.append({
        "Model": model_name,
        "DB": "Chroma",
        "Build Time": build_time
    })

## **Create Qdrant Collections**

Qdrant is a production-grade vector search engine with a rich filtering and payload system. Unlike FAISS, it natively supports metadata storage alongside vectors, and unlike Chroma, it offers fine-grained control over distance metrics and indexing parameters. Running it with `":memory:"` keeps everything in-process, making it fast to set up for benchmarking without requiring a running server.

Initializes an in-memory Qdrant client and builds one collection per embedding model:

- `QdrantClient(":memory:")` - runs Qdrant entirely in RAM with no external server or disk persistence required
- `VectorParams`- explicitly sets the vector `size` from `embeddings.shape[1]` and the distance metric to `Distance.COSINE`
- `delete_collection()` in `try/except` - defensive cleanup before recreating a collection with the same name
- `PointStruct` - Qdrant's unit of insertion; each point carries an integer `id`, the embedding vector, and a `payload` dict containing the raw document text
- `qdrant_client.upsert()` - bulk-inserts all points into the collection; more efficient than adding one point at a time
- `qdrant_collections` stores the collection **name** as a string rather than a client object, since Qdrant queries are issued through the client directly


In [ ]:
qdrant_client = QdrantClient(
    ":memory:"
)

In [ ]:
qdrant_collections = {}

In [ ]:
for model_name, embeddings in embedding_store.items():

    collection_name = (
        f"{model_name.lower()}_qdrant"
    )

    try:
        qdrant_client.delete_collection(
            collection_name
        )
    except:
        pass

    dimension = embeddings.shape[1]

    qdrant_client.create_collection(
        collection_name,
        vectors_config=VectorParams(
            size=dimension,
            distance=Distance.COSINE
        )
    )

    start = time.time()

    points = []

    for idx, emb in enumerate(embeddings):

        points.append(
            PointStruct(
                id=idx,
                vector=emb.tolist(),
                payload={
                    "text": documents[idx]
                }
            )
        )

    qdrant_client.upsert(
        collection_name,
        points
    )

    build_time = time.time() - start

    qdrant_collections[
        model_name
    ] = collection_name

    faiss_metrics.append({
        "Model": model_name,
        "DB": "Qdrant",
        "Build Time": build_time
    })

## **Index Build Time Comparison**

Summarizes and ranks the index construction time across all three vector databases and embedding models.


In [ ]:
build_df = pd.DataFrame(
    faiss_metrics
)

build_df.sort_values(
    by="Build Time"
)

## **Load Embedding Models for Query Encoding**

Document embeddings are already stored.

We now encode the SciFact queries using the same models.

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
EMBEDDING_MODELS = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "BGE": "BAAI/bge-base-en-v1.5",
    "E5": "intfloat/e5-base-v2"
}

In [ ]:
query_embeddings = {}

In [ ]:
for model_name, model_path in EMBEDDING_MODELS.items():

    print(f"Loading {model_name}")

    model = SentenceTransformer(
        model_path,
        device="cuda"
    )

    query_texts = list(queries.values())

    embeddings = model.encode(
        query_texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    query_embeddings[model_name] = embeddings

## **Benchmark FAISS Query Speed**

Extends the latency benchmark to include Qdrant, completing the three-way comparison across all embedding models.

qdrant_client.query_points() - the current SDK method for single-vector search; takes collection_name, a raw query vector as a list, and limit for top-K results
Like Chroma, Qdrant queries are issued one at a time in a loop, so latency reflects per-query overhead rather than batched throughput
All three databases now contribute to latency_results, enabling a unified DataFrame comparison in the next cell

In [ ]:
latency_results = []

In [ ]:
TOP_K = 10

In [ ]:
for model_name in embedding_store:

    index = faiss_indexes[model_name]

    queries_emb = query_embeddings[model_name]

    start = time.time()

    D, I = index.search(
        queries_emb.astype("float32"),
        TOP_K
    )

    latency = (
        time.time() - start
    ) / len(queries_emb)

    latency_results.append({
        "Model": model_name,
        "DB": "FAISS",
        "Latency(ms)": latency * 1000
    })

In [ ]:
for model_name in embedding_store:

    collection = chroma_collections[
        model_name
    ]

    queries_emb = query_embeddings[
        model_name
    ]

    start = time.time()

    for q in queries_emb:

        collection.query(
            query_embeddings=[
                q.tolist()
            ],
            n_results=TOP_K
        )

    latency = (
        time.time() - start
    ) / len(queries_emb)

    latency_results.append({
        "Model": model_name,
        "DB": "Chroma",
        "Latency(ms)": latency * 1000
    })

In [ ]:
for model_name in embedding_store:

    collection_name = qdrant_collections[
        model_name
    ]

    queries_emb = query_embeddings[
        model_name
    ]

    start = time.time()

    for q in queries_emb:

        qdrant_client.query_points(
            collection_name=collection_name,
            query=q.tolist(),
            limit=TOP_K
        )

    latency = (
        time.time() - start
    ) / len(queries_emb)

    latency_results.append({
        "Model": model_name,
        "DB": "Qdrant",
        "Latency(ms)": latency * 1000
    })


## **Query Latency Comparison - FAISS vs Chroma vs Qdrant**

Collects all latency results into a DataFrame and ranks entries from fastest to slowest query response time.


In [ ]:
latency_df = pd.DataFrame(
    latency_results
)

latency_df.sort_values(
    by="Latency(ms)"
)

## **Evaluating Retrieval Quality - Recall**

Recall measures the fraction of relevant documents successfully retrieved within the top 10 results, averaged across all queries. It is the standard metric for evaluating dense retrieval systems against ground truth relevance judgments.

- `recall_at_k()` - computes per-query recall by taking the intersection of the top-k retrieved doc IDs with the ground truth `relevant` set from `qrels`
- `doc_ids[i]` - maps FAISS integer indices back to corpus document ID strings required for matching against `qrels`
- **Chroma** returns string integer IDs in `result["ids"][0]`, so `int(x)` is needed before mapping through `doc_ids`
- **Qdrant** returns `ScoredPoint` objects; `point.id` gives the integer index used at upsert time
- All three loops follow the same pattern: retrieve top-10, map indices to doc IDs, compute recall per query, append mean across all queries to `evaluation_results`



In [ ]:
def recall_at_k(
    retrieved,
    relevant,
    k=10
):

    retrieved = retrieved[:k]

    hits = len(
        set(retrieved).intersection(
            relevant
        )
    )

    return hits / len(relevant)

In [ ]:
evaluation_results = []

In [ ]:
query_ids = list(
    queries.keys()
)

In [ ]:
for model_name in embedding_store:

    recalls = []

    index = faiss_indexes[
        model_name
    ]

    D, I = index.search(
        query_embeddings[
            model_name
        ].astype("float32"),
        10
    )

    for row_idx, query_id in enumerate(
        query_ids
    ):

        relevant_docs = set(
            qrels[query_id].keys()
        )

        retrieved = [
            doc_ids[i]
            for i in I[row_idx]
        ]

        recalls.append(
            recall_at_k(
                retrieved,
                relevant_docs,
                10
            )
        )

    evaluation_results.append({
        "Model": model_name,
        "DB": "FAISS",
        "Recall@10": np.mean(
            recalls
        )
    })

In [ ]:
for model_name in embedding_store:

    recalls = []

    collection = chroma_collections[
        model_name
    ]

    for idx, query_id in enumerate(
        query_ids
    ):

        result = collection.query(
            query_embeddings=[
                query_embeddings[
                    model_name
                ][idx].tolist()
            ],
            n_results=10
        )

        retrieved = [
            doc_ids[int(x)]
            for x in result["ids"][0]
        ]

        relevant_docs = set(
            qrels[query_id].keys()
        )

        recalls.append(
            recall_at_k(
                retrieved,
                relevant_docs
            )
        )

    evaluation_results.append({
        "Model": model_name,
        "DB": "Chroma",
        "Recall@10": np.mean(
            recalls
        )
    })

In [ ]:
for model_name in embedding_store:

    recalls = []

    collection_name = qdrant_collections[
        model_name
    ]

    for idx, query_id in enumerate(
        query_ids
    ):

        response = qdrant_client.query_points(
            collection_name=collection_name,
            query=query_embeddings[
                model_name
            ][idx].tolist(),
            limit=10
        )

        retrieved = [
            doc_ids[point.id]
            for point in response.points
        ]

        relevant_docs = set(
            qrels[query_id].keys()
        )

        recalls.append(
            recall_at_k(
                retrieved,
                relevant_docs
            )
        )

    evaluation_results.append({
        "Model": model_name,
        "DB": "Qdrant",
        "Recall@10": np.mean(
            recalls
        )
    })

## **Recall Rankings - All Models and Databases**

Consolidates evaluation results into a DataFrame ranked by retrieval quality, highest to lowest.


In [ ]:
evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df.sort_values(
    by="Recall@10",
    ascending=False
)

## **Conclusion**

In this notebook, we benchmarked multiple embedding models and vector databases using the BEIR SciFact dataset to evaluate retrieval performance, indexing speed, and query latency.

### **Key Findings**

### **Embedding Model Performance**

The embedding model had the greatest impact on retrieval quality.

| Model | Recall@10 |
|---------|---------|
| BGE Base | 0.8659 |
| E5 Base | 0.8464 |
| MiniLM | 0.7833 |

**BGE Base achieved the highest Recall@10 score**, indicating better semantic understanding and retrieval performance on the SciFact benchmark. E5 Base performed closely behind, while MiniLM provided a faster and more lightweight alternative with lower retrieval accuracy.


### **Vector Database Performance**

The choice of vector database had minimal impact on retrieval quality because all databases operated on the same embeddings and similarity metric.

| Database | Retrieval Quality |
|------------|------------------|
| FAISS | Same |
| Chroma | Same |
| Qdrant | Same |

For this dataset, retrieval quality was determined primarily by the embedding model rather than the vector database.


### **Index Creation Performance**

| Database | Build Time |
|------------|-----------|
| FAISS | Fastest |
| Qdrant | Moderate |
| Chroma | Slowest |

FAISS demonstrated significantly faster index construction compared to Qdrant and Chroma, making it suitable for rapid experimentation and local deployments.



### **Query Latency Performance**

| Database | Observation |
|------------|------------|
| Chroma | Lowest latency in this benchmark |
| FAISS | Comparable low latency |
| Qdrant | Higher latency on small datasets |

On the relatively small SciFact dataset, both FAISS and Chroma delivered very fast query responses, while Qdrant introduced additional overhead associated with production-oriented vector database functionality.


## **Overall Observations**

- Embedding quality has a greater influence on retrieval performance than vector database selection.
- BGE Base produced the most accurate retrieval results among the evaluated models.
- FAISS provided the fastest indexing performance.
- Chroma and FAISS offered similarly low query latency for small-scale datasets.
- Qdrant is designed for production-scale deployments and its advantages become more apparent on larger datasets and distributed environments.


## **Best Performing Combinations**

### **Highest Retrieval Accuracy**
**BGE Base + Any Vector Database**

`Recall@10: **0.8659**`

### **Fastest Index Creation**
`**BGE Base + FAISS**`

### **Best Overall Balance of Accuracy and Performance**
`**BGE Base + FAISS**`

This combination achieved the highest retrieval accuracy while maintaining extremely fast index creation and low query latency.


## **Final Takeaway**

For small to medium-sized Retrieval-Augmented Generation (RAG) workloads, selecting a strong embedding model is often more important than the choice of vector database. In this benchmark, BGE Base consistently delivered the best retrieval performance, while FAISS provided the most efficient indexing and retrieval experience, making the combination of **BGE Base and FAISS** the strongest overall solution.